# FR Y-15 Schedule G: quarterly update generator

Takes the Schedule G supporting workbook (the one with `Sheet1`, `FR Y-15 Schedule G` and
`QoQ Notes & Commentary` tabs) and produces two files:

1. **Analysis workbook (.xlsx)**: Summary, Detail, Inputs and Notes tabs, with live formulas.
2. **Treasury update (.docx)**: narrative by tier, tier summary table and line/maturity detail table.

**Before you run**
1. Upload the workbook to a Unity Catalog Volume: *Catalog* > your catalog > schema > volume > *Upload to this volume*.
2. Fill in the widgets at the top of the notebook (input path, output folder, quarter labels).
3. Run all cells. Outputs are written to the output folder in the same volume (download from Catalog Explorer).

**Notes**
- The workbook must have cached values (saved from Excel, not from a script). The notebook reads values, not formulas.
- Line items are located by RISK code (RISKY838 to RISKY896), not by row number, so inserted rows will not break it.
- Narrative text is generated from the numbers. Review it before sending. Add quarter-specific context in the
  `MANUAL_COMMENTARY` and `MANUAL_FOLLOW_UPS` settings below.

## 1. Install libraries
`openpyxl` is usually preinstalled on Databricks runtimes. `python-docx` often is not. If `%pip` is blocked on your
cluster, ask IT to add both packages as cluster libraries.

In [0]:
%pip install -q openpyxl python-docx

In [0]:
dbutils.library.restartPython()

## 2. Settings
The first run creates the widgets at the top of the notebook and stops with a message. Fill in the input path,
output folder and quarter labels, then **Run all** again.

In [0]:
import os, re, shutil, tempfile, datetime

def _widget(name, default, label=None):
    """Databricks widget when running in a notebook; environment variable otherwise (for local testing)."""
    try:
        dbutils.widgets.text(name, default, label or name)
        return dbutils.widgets.get(name).strip()
    except NameError:
        return os.environ.get(name.upper(), default)

INPUT_PATH = _widget("input_path", "/Volumes/<catalog>/<schema>/<volume>/your_workbook.xlsx", "Input workbook (.xlsx)")
OUTPUT_DIR = _widget("output_dir", "/Volumes/<catalog>/<schema>/<volume>/schedule_g_output", "Output folder")
Q1 = _widget("q1_label", "1Q26", "Oldest quarter label")
Q2 = _widget("q2_label", "2Q26", "Prior quarter label")
Q3 = _widget("q3_label", "3Q26", "Current quarter label")
AS_OF = _widget("as_of", "September 30, 2026", "As-of date (text)")
ENTITY = _widget("entity", "Pinnacle Financial Partners", "Entity name")

GREETING = "Good Afternoon,"
SIGN_OFF = ["Thank you,", "[Name]"]

# Quarter-specific context that cannot be derived from the numbers. Keyed by Y-15 line item. Appended to that
# line's 30-days-or-less bullet in the document and the Detail tab. Review each quarter.
MANUAL_COMMENTARY = {
    "1.b": "Brokered NMDs continued to renew ahead of maturity, and brokered CDs previously in the 365+ day bucket "
           "continued to age into shorter buckets.",
}
# Extra follow-up items to append to the auto-generated list.
MANUAL_FOLLOW_UPS = [
    "The maturity profile and issuance activity of the FHLB advances driving any change in the longer-dated buckets.",
]

# Source layout (change only if the workbook layout changes)
SRC_SHEET, WEIGHT_SHEET, QOQ_SHEET = "Sheet1", "FR Y-15 Schedule G", "QoQ Notes & Commentary"
COL_CODE, COL_Q1, COL_Q2, COL_Q3 = 1, 3, 4, 6            # Sheet1 columns A, C, D, F
QOQ_COL_LABEL, QOQ_COL_Q2, QOQ_COL_Q3, QOQ_COL_NOTE = 3, 13, 19, 30   # QoQ tab: C, M:P, S:V, AD

# Thresholds
COMPONENT_MIN_K = 10_000    # show a component mover if its change is at least $10MM (amounts are in $ thousands)
LONG_BUCKET_MIN_K = 10_000  # call out longer-dated buckets only if some bucket moves at least $10MM

if "<catalog>" in INPUT_PATH or "<catalog>" in OUTPUT_DIR:
    raise ValueError("Set the input_path and output_dir widgets (replace the <catalog>/<schema>/<volume> placeholders).")

## 3. Load and parse the workbook

In [0]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.formatting.rule import FormulaRule
from openpyxl.comments import Comment
from openpyxl.chart import BarChart, Reference
from openpyxl.utils import get_column_letter as L
from openpyxl.worksheet.properties import PageSetupProperties
from openpyxl.workbook.properties import CalcProperties

MATURITIES = ["30 days or less", "31-90 days", "91-180 days", "181-365 days"]
TIERS = ["First tier", "Second tier", "Third tier", "All other"]
LINES = [  # tier, Y-15 item, description, first RISK code (four consecutive codes, one per maturity bucket)
    ("First tier",  "1.a", "Funding secured by Level 1 liquid assets", 838),
    ("First tier",  "1.b", "Retail brokered deposits and sweeps", 842),
    ("First tier",  "1.c", "Unsecured wholesale funding, outside the financial sector", 846),
    ("Second tier", "2.a", "Funding secured by Level 2A liquid assets", 858),
    ("Third tier",  "3.a", "Funding secured by Level 2B liquid assets", 870),
    ("Third tier",  "3.c", "Unsecured wholesale funding, within the financial sector", 878),
    ("All other",   "4",   "All other components of short-term wholesale funding", 886),
]
ITEM6_CODE, RWA_CODE, RATIO_CODE = 894, 895, 896
DEFAULT_WEIGHTS = {"First tier": [.25, .10, 0, 0], "Second tier": [.5, .25, .10, 0],
                   "Third tier": [.75, .5, .25, .10], "All other": [1, .75, .5, .25]}

def num(x):
    return float(x) if isinstance(x, (int, float)) and not isinstance(x, bool) else 0.0

def to_local(path):
    """openpyxl needs random access. Copy from Volumes/DBFS/Workspace to local disk first."""
    if path.startswith(("/Volumes", "/dbfs", "/Workspace")):
        tmp = os.path.join(tempfile.mkdtemp(), os.path.basename(path))
        shutil.copy(path, tmp)
        return tmp
    return path

def load_source(path):
    wb = openpyxl.load_workbook(to_local(path), data_only=True)
    # --- quarterly values by RISK code
    ws = wb[SRC_SHEET]
    data = {}
    for r in range(1, ws.max_row + 1):
        m = re.match(r"^RISKY(\d+)$", str(ws.cell(r, COL_CODE).value or "").strip())
        if m:
            data[int(m.group(1))] = (num(ws.cell(r, COL_Q1).value), num(ws.cell(r, COL_Q2).value), num(ws.cell(r, COL_Q3).value))
    needed = [c + k for _, _, _, c in LINES for k in range(4)] + [ITEM6_CODE, RWA_CODE]
    missing = [f"RISKY{c}" for c in needed if c not in data]
    if missing:
        raise ValueError(f"RISK codes not found in {SRC_SHEET} column A: {missing}")
    # --- weights (read from the Y-15 tab so they track the filing; fall back to defaults)
    weights = {t: list(w) for t, w in DEFAULT_WEIGHTS.items()}
    if WEIGHT_SHEET in wb.sheetnames:
        g = wb[WEIGHT_SHEET]
        for tier, row in zip(TIERS, (12, 16, 21, 22)):
            vals = [g.cell(row, c).value for c in (12, 14, 16, 18)]
            if all(isinstance(v, (int, float)) for v in vals):
                weights[tier] = [float(v) for v in vals]
    # --- line components from the QoQ tab
    comps = load_components(wb) if QOQ_SHEET in wb.sheetnames else {}
    return data, weights, comps

def block_item(label):
    m = re.match(r"^\s*(\w+)\s+tier\s*\|\s*line\s*([A-Za-z])\s*$", str(label or ""), re.I)
    if not m:
        return None
    w, ln = m.group(1).lower(), m.group(2).lower()
    if w.startswith("first"):  return f"1.{ln}"
    if w.startswith("second"): return f"2.{ln}"
    if w.startswith("th"):     return f"3.{ln}"      # tab spells it "Thrid"
    if w.startswith("four"):   return "4"
    return None

def short_name(label):
    s = str(label).strip()
    lv = re.search(r"level\s+(\w+)", s, re.I)
    low = s.lower()
    if low.startswith("non-public funds secured by non hqla"): return "Non-public funds (non-HQLA)"
    if low.startswith("public funds cds secured by non hqla"): return "Public fund CDs (non-HQLA)"
    if low.startswith("public funds secured by non hqla"):     return "Public funds (non-HQLA)"
    if low.startswith("repurchase agreements secured by non hqla"): return "Repos (non-HQLA)"
    if low.startswith("non-public funds secured by level") and lv: return f"Non-public funds (Level {lv.group(1)})"
    if low.startswith("public funds secured by level") and lv:     return f"Public funds (Level {lv.group(1)})"
    if low.startswith("repurchase agreements secured by level") and lv: return f"Repos (Level {lv.group(1)})"
    if low.startswith("non-operational nmds"): return "Non-operational NMDs"
    if low.startswith("reciprocal"):           return "Reciprocal (ICS) deposits"
    if low.startswith("cds obtained"):         return "CDs"
    if low.startswith("non secured portion"):  return "Non-secured portion of secured funding"
    if low.startswith("long term debt"):       return "Long-term debt and notes payable"
    return s[:45]

def load_components(wb):
    """Return {item: [ {name, q2:[4], q3:[4], note} ]}. The last row of each block is the line total and is dropped."""
    q = wb[QOQ_SHEET]
    out, cur, rows = {}, None, []
    def finish():
        if cur and len(rows) > 1:
            out[cur] = rows[:-1]
    for r in range(1, q.max_row + 1):
        lab = q.cell(r, QOQ_COL_LABEL).value
        item = block_item(lab)
        if item:
            finish(); cur, rows = item, []
        elif cur and lab not in (None, ""):
            rows.append(dict(name=short_name(lab),
                             q2=[num(q.cell(r, QOQ_COL_Q2 + k).value) for k in range(4)],
                             q3=[num(q.cell(r, QOQ_COL_Q3 + k).value) for k in range(4)],
                             note=q.cell(r, QOQ_COL_NOTE).value))
        elif cur and rows:
            finish(); cur, rows = None, []
    finish()
    return out

data, WEIGHTS, COMPS = load_source(INPUT_PATH)
print(f"Loaded {len(data)} RISK codes; components found for lines: {sorted(COMPS)}")

## 4. Compute the analysis

In [0]:
def pct_chg(new, old):
    return None if old == 0 else new / old - 1

ROWS = []
for tier, item, desc, c0 in LINES:
    for k, mat in enumerate(MATURITIES):
        q1, q2, q3 = data[c0 + k]
        w = WEIGHTS[tier][k]
        ROWS.append(dict(tier=tier, item=item, line=desc, mat=mat, k=k, code=f"RISKY{c0 + k}", w=w,
                         q1=q1, q2=q2, q3=q3, d12=q2 - q1, d23=q3 - q2, wd=w * (q3 - q2)))

def wsum(rows, q):  return sum(r["w"] * r[q] for r in rows)
def usum(rows, q):  return sum(r[q] for r in rows)

TOTAL = {q: wsum(ROWS, q) for q in ("q1", "q2", "q3")}
TOTAL_WD = TOTAL["q3"] - TOTAL["q2"]
TIER_AGG = {t: {q: wsum([r for r in ROWS if r["tier"] == t], q) for q in ("q1", "q2", "q3")} for t in TIERS}
TIER_UNW = {t: {q: usum([r for r in ROWS if r["tier"] == t], q) for q in ("q2", "q3")} for t in TIERS}
BUCKET_AGG = {m: {q: wsum([r for r in ROWS if r["mat"] == m], q) for q in ("q1", "q2", "q3")} for m in MATURITIES}
UNW = {q: usum(ROWS, q) for q in ("q1", "q2", "q3")}

REPORTED = dict(zip(("q1", "q2", "q3"), data[ITEM6_CODE]))
RWA = dict(zip(("q1", "q2", "q3"), data[RWA_CODE]))
RATIO_REPORTED = [v / 100 for v in data.get(RATIO_CODE, (0, 0, 0))[:2]]

print("Tie-out of computed item 6 to reported item 6 ($K):")
for q, lab in (("q1", Q1), ("q2", Q2), ("q3", Q3)):
    diff = TOTAL[q] - REPORTED[q]
    flag = "OK" if abs(diff) <= 2 else "CHECK WEIGHTS / MAPPING"
    print(f"  {lab}: computed {TOTAL[q]:,.0f} | reported {REPORTED[q]:,.0f} | diff {diff:,.2f} | {flag}")
if RWA["q3"] <= 0:
    print(f"WARNING: {Q3} average RWA (RISKY{RWA_CODE}) is blank, so the {Q3} ratio cannot be computed.")

## 5. Build the analysis workbook

In [0]:
def money(k):
    mm = abs(k) / 1000.0
    if mm >= 1000: return f"${mm/1000:.2f}B"
    if mm < 1:     return f"${mm:,.1f}MM"
    return f"${mm:,.0f}MM"

def smoney(k):
    return ("+" if k >= 0 else "-") + money(k)

def ud(k):
    """'up $44MM' / 'down $18MM'. Words avoid a sign wrapping away from its amount at a line break."""
    return ("up " if k >= 0 else "down ") + money(k)

def fpct(p):
    return "new" if p is None else f"{p:+.1%}"

def movers(item, k_buckets, min_k=COMPONENT_MIN_K, limit=4):
    """Net change by component over the given maturity buckets, largest first."""
    res = []
    for c in COMPS.get(item, []):
        d = sum(c["q3"][k] - c["q2"][k] for k in k_buckets)
        if abs(d) >= min_k:
            res.append((c["name"], d))
    res.sort(key=lambda x: -abs(x[1]))
    return res[:limit]

def excel_comment(item, k):
    parts = [f"{n} {ud(d)}" for n, d in movers(item, [k])]
    txt = "; ".join(parts)
    if k == 0 and item in MANUAL_COMMENTARY:
        txt = (txt + ". " if txt else "") + MANUAL_COMMENTARY[item]
    return txt

AR = "Arial"
f_base = Font(name=AR, size=10); f_b = Font(name=AR, size=10, bold=True)
f_title = Font(name=AR, size=14, bold=True, color="1F3864"); f_sub = Font(name=AR, size=9, italic=True, color="595959")
f_hdr = Font(name=AR, size=10, bold=True, color="FFFFFF"); f_in = Font(name=AR, size=10, color="0000FF")
f_grey = Font(name=AR, size=8, color="808080"); f_note = Font(name=AR, size=9, color="404040")
fill_hdr = PatternFill("solid", fgColor="1F3864"); fill_sec = PatternFill("solid", fgColor="2F5597")
fill_tot = PatternFill("solid", fgColor="D9E2F3"); fill_in = PatternFill("solid", fgColor="FFFF00")
tier_fill = {"First tier": "FFFFFF", "Second tier": "F2F2F2", "Third tier": "FFFFFF", "All other": "F2F2F2"}
thin = Side(style="thin", color="BFBFBF"); top_b = Border(top=Side(style="thin", color="000000"))
NUM = '#,##0;(#,##0);"-"'; PCT = '0.0%;(0.0%);"-"'; PCT2 = "0.00%"; WGT = "0%"
ctr = Alignment(horizontal="center", vertical="center", wrap_text=True)
rgt = Alignment(horizontal="right"); wrap = Alignment(wrap_text=True, vertical="top")

def put(ws, ref, val, font=f_base, fmt=None, fill=None, al=None, border=None):
    c = ws[ref]; c.value = val; c.font = font
    if fmt: c.number_format = fmt
    if fill: c.fill = fill
    if al: c.alignment = al
    if border: c.border = border
    return c

def build_excel(out_path):
    wb = openpyxl.Workbook()
    S = wb.active; S.title = "Summary"
    D = wb.create_sheet("Detail"); I = wb.create_sheet("Inputs"); N = wb.create_sheet("Notes")
    for ws in (S, D, I, N): ws.sheet_view.showGridLines = False

    # ---------- Inputs
    put(I, "A1", "Inputs and reference data", f_title)
    put(I, "A2", "Blue = hardcoded input. Yellow fill = cell to update. All amounts in $ thousands unless noted.", f_sub)
    put(I, "A4", "A. Item 5 maturity weights", f_b)
    put(I, "A5", f"Source: {WEIGHT_SHEET} tab, cells L12:R22.", f_sub)
    put(I, "A6", "Tier", f_hdr, fill=fill_hdr)
    for j, m in enumerate(MATURITIES): put(I, f"{L(2+j)}6", m, f_hdr, fill=fill_hdr, al=ctr)
    for i, t in enumerate(TIERS):
        put(I, f"A{7+i}", t)
        for j in range(4): put(I, f"{L(2+j)}{7+i}", WEIGHTS[t][j], f_in, WGT)
    put(I, "A12", "B. Review-flag thresholds (Detail tab, Flag column)", f_b)
    put(I, "A13", "Flag when absolute QoQ % change is at least"); put(I, "B13", 0.25, f_in, "0%", fill_in)
    put(I, "A14", "and absolute QoQ $ change is at least ($K)"); put(I, "B14", 50000, f_in, "#,##0", fill_in)
    put(I, "C14", "Equals $50 million. Both tests must be met; this screens out large % moves on tiny balances.", f_sub)
    put(I, "A16", f"C. Reported figures used for tie-out (source: {SRC_SHEET})", f_b)
    put(I, "A17", "Item", f_hdr, fill=fill_hdr)
    for j, q in enumerate([Q1, Q2, Q3]): put(I, f"{L(2+j)}17", q, f_hdr, fill=fill_hdr, al=ctr)
    put(I, "A18", f"Total STWF, item 6 (RISKY{ITEM6_CODE})")
    for j, q in enumerate(("q1", "q2", "q3")): put(I, f"{L(2+j)}18", REPORTED[q], f_in, NUM)
    put(I, "A19", f"Reported ratio, item 8 (RISKY{RATIO_CODE})")
    put(I, "B19", RATIO_REPORTED[0], f_in, PCT2); put(I, "C19", RATIO_REPORTED[1], f_in, PCT2); put(I, "D19", "n/a", f_in, al=rgt)
    I.column_dimensions["A"].width = 48
    for c in "BCDE": I.column_dimensions[c].width = 16

    # ---------- Detail
    put(D, "A1", f"Schedule G line-level detail: {Q1}, {Q2}, {Q3}", f_title)
    put(D, "A2", "$ thousands. Weight = Y-15 item 5 maturity weight (Inputs tab). Weighted change = weight x QoQ $ change, "
                 "i.e. the line's contribution to the change in total short-term wholesale funding.", f_sub)
    heads = ["Tier", "Item", "Line item", "Remaining maturity", "RISK code", "Weight", Q1, Q2, Q3,
             f"{Q1} to {Q2}\n$ chg", f"{Q1} to {Q2}\n% chg", f"{Q2} to {Q3}\n$ chg", f"{Q2} to {Q3}\n% chg",
             f"{Q2} to {Q3}\nweighted $ chg", "% of total\nweighted chg", "Flag",
             "Commentary (derived from the QoQ tab and settings; owner to confirm)"]
    for j, h in enumerate(heads): put(D, f"{L(1+j)}5", h, f_hdr, fill=fill_hdr, al=ctr)
    D.row_dimensions[5].height = 42
    put(D, "G4", "Balances", f_b); put(D, "J4", "Q-o-Q change", f_b)
    FIRST = 6
    for n, r_ in enumerate(ROWS):
        r = FIRST + n
        put(D, f"A{r}", r_["tier"]); put(D, f"B{r}", r_["item"], al=Alignment(horizontal="center"))
        put(D, f"C{r}", r_["line"]); put(D, f"D{r}", r_["mat"]); put(D, f"E{r}", r_["code"], f_grey)
        put(D, f"F{r}", f"=INDEX(Inputs!$B$7:$E$10,MATCH($A{r},Inputs!$A$7:$A$10,0),MATCH($D{r},Inputs!$B$6:$E$6,0))", f_base, WGT)
        for col, key in (("G", "q1"), ("H", "q2"), ("I", "q3")): put(D, f"{col}{r}", r_[key], f_in, NUM)
        put(D, f"J{r}", f"=H{r}-G{r}", f_base, NUM)
        put(D, f"K{r}", f'=IF(AND(G{r}=0,H{r}=0),"-",IF(G{r}=0,"new",H{r}/G{r}-1))', f_base, PCT, al=rgt)
        put(D, f"L{r}", f"=I{r}-H{r}", f_base, NUM)
        put(D, f"M{r}", f'=IF(AND(H{r}=0,I{r}=0),"-",IF(H{r}=0,"new",I{r}/H{r}-1))', f_base, PCT, al=rgt)
        put(D, f"N{r}", f"=F{r}*L{r}", f_base, NUM)
        put(D, f"P{r}", f'=IF(AND(ISNUMBER(M{r}),ABS(M{r})>=Inputs!$B$13,ABS(L{r})>=Inputs!$B$14),"Review",'
                        f'IF(AND(M{r}="new",ABS(L{r})>=Inputs!$B$14),"Review",""))', f_b, al=Alignment(horizontal="center"))
        put(D, f"Q{r}", excel_comment(r_["item"], r_["k"]), f_note)
        for cc in range(1, 18):
            D.cell(r, cc).fill = PatternFill("solid", fgColor=tier_fill[r_["tier"]])
            if r_["k"] == 0: D.cell(r, cc).border = Border(top=thin)
    LAST = FIRST + len(ROWS) - 1; TW, TU = LAST + 1, LAST + 2
    for r in range(FIRST, LAST + 1):
        put(D, f"O{r}", f"=IF($N${TW}=0,0,N{r}/$N${TW})", f_base, PCT)
    put(D, f"A{TW}", "Total short-term wholesale funding (weighted, item 6)", f_b)
    put(D, f"A{TU}", "Total, unweighted gross balances", f_b)
    for col in "GHI":
        put(D, f"{col}{TW}", f"=SUMPRODUCT($F${FIRST}:$F${LAST},{col}{FIRST}:{col}{LAST})", f_b, NUM)
        put(D, f"{col}{TU}", f"=SUM({col}{FIRST}:{col}{LAST})", f_b, NUM)
    for rr in (TW, TU):
        put(D, f"J{rr}", f"=H{rr}-G{rr}", f_b, NUM); put(D, f"K{rr}", f'=IF(G{rr}=0,"-",H{rr}/G{rr}-1)', f_b, PCT, al=rgt)
        put(D, f"L{rr}", f"=I{rr}-H{rr}", f_b, NUM); put(D, f"M{rr}", f'=IF(H{rr}=0,"-",I{rr}/H{rr}-1)', f_b, PCT, al=rgt)
        for cc in range(1, 18):
            D.cell(rr, cc).fill = fill_tot
            D.cell(rr, cc).border = top_b if rr == TW else Border()
    put(D, f"N{TW}", f"=SUM(N{FIRST}:N{LAST})", f_b, NUM); put(D, f"O{TW}", f"=SUM(O{FIRST}:O{LAST})", f_b, PCT)
    put(D, f"A{TU+2}", f"Lines with no balance (1.d, 2.b, 3.b) are omitted. Blue = values read from {SRC_SHEET} of the source workbook.", f_sub)
    for j, w in enumerate([12, 6, 52, 16, 10, 8, 13, 13, 13, 13, 10, 13, 10, 15, 12, 9, 95]): D.column_dimensions[L(1+j)].width = w
    D.freeze_panes = f"D{FIRST}"; D.auto_filter.ref = f"A5:Q{LAST}"
    D.conditional_formatting.add(f"A{FIRST}:Q{LAST}", FormulaRule(formula=[f'$P{FIRST}="Review"'],
                                 fill=PatternFill("solid", bgColor="FFE699", fgColor="FFE699")))

    # ---------- Summary
    rng = lambda col: f"Detail!${col}${FIRST}:${col}${LAST}"
    put(S, "A1", f"FR Y-15 Schedule G | Short-term wholesale funding | {Q3} quarter-over-quarter analysis", f_title)
    put(S, "A2", f"{ENTITY}  |  $ thousands unless noted  |  as of {AS_OF}  |  generated {datetime.date.today():%Y-%m-%d}", f_sub)
    def section(row, text, ncol=10):
        for c in range(1, ncol + 1): S.cell(row, c).fill = fill_sec
        put(S, f"A{row}", text, Font(name=AR, size=10, bold=True, color="FFFFFF"), fill=fill_sec)
    def header(row, labels):
        for j, h in enumerate(labels): put(S, f"{L(1+j)}{row}", h, f_hdr, fill=fill_hdr, al=ctr if j else Alignment(vertical="center"))
        S.row_dimensions[row].height = 30
    chg = f"{Q2} to {Q3}"
    section(4, "1. Headline")
    header(5, ["Metric", Q1, Q2, Q3, f"{chg} $ chg", f"{chg} % chg", f"{Q3} vs {Q1} % chg"])
    put(S, "A6", "Total short-term wholesale funding (item 6)", f_b)
    for col, dc in (("B", "G"), ("C", "H"), ("D", "I")): put(S, f"{col}6", f"=Detail!{dc}{TW}", f_b, NUM)
    put(S, "E6", "=D6-C6", f_b, NUM); put(S, "F6", "=D6/C6-1", f_b, PCT); put(S, "G6", "=D6/B6-1", f_b, PCT)
    put(S, "A7", "   of which remaining maturity 30 days or less")
    for col, dc in (("B", "G"), ("C", "H"), ("D", "I")):
        put(S, f"{col}7", f'=SUMPRODUCT(({rng("D")}=$A$26)*{rng("F")}*{rng(dc)})', f_base, NUM)
    put(S, "E7", "=D7-C7", f_base, NUM); put(S, "F7", "=D7/C7-1", f_base, PCT); put(S, "G7", "=D7/B7-1", f_base, PCT)
    put(S, "A8", "   30 days or less as % of total")
    for col in "BCD": put(S, f"{col}8", f"={col}7/{col}6", f_base, PCT)
    put(S, "A9", "Average risk-weighted assets (item 7)")
    put(S, "B9", RWA["q1"], f_in, NUM); put(S, "C9", RWA["q2"], f_in, NUM)
    put(S, "D9", RWA["q3"] if RWA["q3"] > 0 else None, f_in, NUM, fill_in)
    put(S, "E9", '=IF(D9>0,D9-C9,"")', f_base, NUM); put(S, "F9", '=IF(D9>0,D9/C9-1,"")', f_base, PCT); put(S, "G9", '=IF(D9>0,D9/B9-1,"")', f_base, PCT)
    if RWA["q3"] <= 0:
        S["D9"].comment = Comment(f"{Q3} average RWA was blank in the source workbook. Enter it here ($ thousands) and the ratio below updates.", "Analysis")
    put(S, "A10", "Short-term wholesale funding ratio (item 8)", f_b)
    put(S, "B10", "=B6/B9", f_b, PCT2); put(S, "C10", "=C6/C9", f_b, PCT2); put(S, "D10", '=IF(D9>0,D6/D9,"Pending RWA")', f_b, PCT2, al=rgt)
    put(S, "E10", '=IF(D9>0,(D10-C10)*10000,"")', f_base, '0" bps";(0)" bps"')
    put(S, "A11", f"   Illustrative: {Q3} ratio if {Q3} average RWA equals {Q2}", f_note); put(S, "D11", "=D6/C9", f_note, PCT2)
    put(S, "A12", f"   Average RWA needed for the ratio to hold at the {Q2} level", f_note); put(S, "D12", "=D6/C10", f_note, NUM)
    put(S, "F12", "=D12/C9-1", f_note, PCT); put(S, "G12", f"<- % RWA growth required vs {Q2}", f_note)
    put(S, "A13", f"Yellow cell (D9) is the open input when {Q3} average RWA is not in the source workbook.", f_sub)

    section(15, "2. What drove the change, by tier (weighted = item 5 basis)")
    header(16, ["Tier", f"{Q1} weighted", f"{Q2} weighted", f"{Q3} weighted", f"{chg} $ chg", f"{chg} % chg",
                "% of total weighted chg", f"{Q2} unweighted", f"{Q3} unweighted", "Unweighted % chg"])
    for i, t in enumerate(TIERS):
        r = 17 + i; put(S, f"A{r}", t)
        for col, dc in (("B", "G"), ("C", "H"), ("D", "I")):
            put(S, f"{col}{r}", f'=SUMPRODUCT(({rng("A")}=$A{r})*{rng("F")}*{rng(dc)})', f_base, NUM)
        put(S, f"E{r}", f"=D{r}-C{r}", f_base, NUM); put(S, f"F{r}", f'=IF(C{r}=0,"-",D{r}/C{r}-1)', f_base, PCT, al=rgt)
        put(S, f"G{r}", f"=IF($E$21=0,0,E{r}/$E$21)", f_base, PCT)
        put(S, f"H{r}", f'=SUMIFS({rng("H")},{rng("A")},$A{r})', f_base, NUM); put(S, f"I{r}", f'=SUMIFS({rng("I")},{rng("A")},$A{r})', f_base, NUM)
        put(S, f"J{r}", f'=IF(H{r}=0,"-",I{r}/H{r}-1)', f_base, PCT, al=rgt)
    put(S, "A21", "Total", f_b)
    for col in "BCDEHI": put(S, f"{col}21", f"=SUM({col}17:{col}20)", f_b, NUM)
    put(S, "F21", "=D21/C21-1", f_b, PCT); put(S, "G21", "=SUM(G17:G20)", f_b, PCT); put(S, "J21", "=I21/H21-1", f_b, PCT)
    for c in range(1, 11): S.cell(21, c).fill = fill_tot; S.cell(21, c).border = top_b
    put(S, "A22", "Weighted tier totals sum to item 6. Unweighted = gross balances across all four maturity buckets.", f_sub)

    section(24, "3. By remaining maturity (item 5, weighted)")
    header(25, ["Remaining maturity", Q1, Q2, Q3, f"{chg} $ chg", f"{chg} % chg", f"{Q3} mix %"])
    for i, m in enumerate(MATURITIES):
        r = 26 + i; put(S, f"A{r}", m)
        for col, dc in (("B", "G"), ("C", "H"), ("D", "I")):
            put(S, f"{col}{r}", f'=SUMPRODUCT(({rng("D")}=$A{r})*{rng("F")}*{rng(dc)})', f_base, NUM)
        put(S, f"E{r}", f"=D{r}-C{r}", f_base, NUM); put(S, f"F{r}", f'=IF(C{r}=0,"-",D{r}/C{r}-1)', f_base, PCT, al=rgt)
        put(S, f"G{r}", f"=D{r}/$D$30", f_base, PCT)
    put(S, "A30", "Total", f_b)
    for col in "BCDE": put(S, f"{col}30", f"=SUM({col}26:{col}29)", f_b, NUM)
    put(S, "F30", "=D30/C30-1", f_b, PCT); put(S, "G30", "=SUM(G26:G29)", f_b, PCT)
    for c in range(1, 8): S.cell(30, c).fill = fill_tot; S.cell(30, c).border = top_b

    section(32, f"4. Largest weighted drivers, {chg} (re-ranks automatically from the Detail tab)")
    def drivers(start, title, fn):
        put(S, f"A{start}", title, f_b)
        header(start + 1, ["Line | maturity", Q2, Q3, "$ chg (unweighted)", "% chg", "Weighted $ chg", "% of total weighted chg", "Detail row (helper)"])
        for k in range(1, 6):
            r = start + 1 + k
            put(S, f"F{r}", f'={fn}({rng("N")},{k})', f_base, NUM)
            put(S, f"H{r}", f'=MATCH(F{r},{rng("N")},0)', f_grey)
            idx = lambda col: f"INDEX({rng(col)},$H{r})"
            put(S, f"A{r}", f'={idx("B")}&" "&{idx("C")}&"  |  "&{idx("D")}', f_base)
            put(S, f"B{r}", f'={idx("H")}', f_base, NUM); put(S, f"C{r}", f'={idx("I")}', f_base, NUM)
            put(S, f"D{r}", f'={idx("L")}', f_base, NUM); put(S, f"E{r}", f'={idx("M")}', f_base, PCT, al=rgt)
            put(S, f"G{r}", f"=IF($E$21=0,0,F{r}/$E$21)", f_base, PCT)
    drivers(33, "Largest increases", "LARGE"); drivers(41, "Largest decreases", "SMALL")
    put(S, "A48", "Weighted change = Y-15 weight x $ change. It ranks lines by their effect on item 6, not by raw balance movement.", f_sub)

    section(50, "5. Tie-out checks")
    header(51, ["Check", Q1, Q2, Q3])
    put(S, "A52", "Computed total (weights x line balances)"); put(S, "A53", "Reported item 6 (source workbook, via Inputs tab)")
    put(S, "A54", "Difference ($K)"); put(S, "A55", "Status (tolerance $2K for whole-$K rounding)")
    put(S, "A56", "Computed ratio vs reported item 8 (difference, pp)")
    for col in "BCD":
        put(S, f"{col}52", f"={col}6", f_base, NUM); put(S, f"{col}53", f"=Inputs!{col}18", f_base, NUM)
        put(S, f"{col}54", f"={col}52-{col}53", f_base, '#,##0.00;(#,##0.00);"-"')
        put(S, f"{col}55", f'=IF(ABS({col}54)<=2,"OK","CHECK")', f_b, al=rgt)
    put(S, "B56", "=(B10-Inputs!B19)*100", f_base, '0.000;(0.000);"-"'); put(S, "C56", "=(C10-Inputs!C19)*100", f_base, '0.000;(0.000);"-"')
    put(S, "D56", "n/a", f_base, al=rgt)
    S.conditional_formatting.add("B55:D55", FormulaRule(formula=['B55="CHECK"'], fill=PatternFill("solid", bgColor="F8CBAD", fgColor="F8CBAD")))
    S.conditional_formatting.add("B55:D55", FormulaRule(formula=['B55="OK"'], fill=PatternFill("solid", bgColor="C6E0B4", fgColor="C6E0B4")))
    S.column_dimensions["A"].width = 76
    for c in "BCDEFGHIJ": S.column_dimensions[c].width = 16
    S.freeze_panes = "B4"

    ch = BarChart(); ch.type = "col"; ch.grouping = "stacked"; ch.overlap = 100; ch.title = "Weighted STWF by tier ($K)"; ch.style = 10
    ch.add_data(Reference(S, min_col=1, max_col=4, min_row=17, max_row=20), from_rows=True, titles_from_data=True)
    ch.set_categories(Reference(S, min_col=2, max_col=4, min_row=5, max_row=5)); ch.height = 7.5; ch.width = 15
    ch.y_axis.numFmt = "#,##0"; ch.y_axis.delete = False; ch.x_axis.delete = False
    S.add_chart(ch, "L4")
    ch2 = BarChart(); ch2.type = "bar"; ch2.title = f"{chg} weighted $ change by tier ($K)"; ch2.style = 10
    ch2.add_data(Reference(S, min_col=5, min_row=16, max_row=20), titles_from_data=True)
    ch2.set_categories(Reference(S, min_col=1, min_row=17, max_row=20)); ch2.height = 7.5; ch2.width = 15; ch2.legend = None
    ch2.y_axis.numFmt = "#,##0"; ch2.y_axis.delete = False; ch2.x_axis.delete = False
    S.add_chart(ch2, "L20")

    # ---------- Notes
    put(N, "A1", "Notes, data flags and open items", f_title)
    notes = [
        ("Scope", f"Built from {SRC_SHEET} of the source workbook ({os.path.basename(INPUT_PATH)}). Amounts are $ thousands. "
                  f"Line items are located by RISK code. Formulas recalculate when the file is opened in Excel."),
        ("Weights", f"Item 5 maturity weights are read from the {WEIGHT_SHEET} tab and reproduce reported item 6 within rounding "
                    f"(see Summary section 5)."),
    ]
    if RWA["q3"] <= 0:
        t2, t3 = TOTAL["q2"], TOTAL["q3"]; ratio2 = t2 / RWA["q2"]
        notes.append((f"{Q3} ratio is not computable yet",
                      f"{Q3} average RWA (item 7) is blank in the source workbook. Enter it in Summary!D9 and the ratio calculates. "
                      f"Total STWF changed {t3/t2-1:+.1%} QoQ, so average RWA needs to move about {t3/t2-1:+.1%} "
                      f"(to roughly ${t3/ratio2/1e6:.1f}B) for the ratio to stay at {ratio2:.2%}."))
    notes += [
        ("Small-base percentages", "Large % moves on small balances are noise. The Detail flag requires both the % and $ thresholds on the Inputs tab."),
        ("Lines omitted", "1.d, 2.b and 3.b carry no balances and are left out of Detail. They would not change any total."),
        ("Commentary column", "Entries on Detail list the largest component movers from the QoQ Notes & Commentary tab plus any text in "
                              "MANUAL_COMMENTARY. They describe what moved, not why. Owners should confirm before they go external."),
    ]
    put(N, "A3", "Topic", f_hdr, fill=fill_hdr); put(N, "B3", "Note", f_hdr, fill=fill_hdr)
    for i, (a, b) in enumerate(notes):
        r = 4 + i; put(N, f"A{r}", a, f_b, al=wrap); put(N, f"B{r}", b, f_base, al=wrap)
        N.row_dimensions[r].height = max(30, 15 * (len(b) // 105 + 1))
    N.column_dimensions["A"].width = 34; N.column_dimensions["B"].width = 115

    for ws in (S, D, I, N):
        ws.sheet_properties.tabColor = "1F3864"
        ws.page_setup.orientation = "landscape"; ws.page_setup.fitToWidth = 1; ws.page_setup.fitToHeight = 0
        ws.sheet_properties.pageSetUpPr = PageSetupProperties(fitToPage=True)
    wb.calculation = CalcProperties(fullCalcOnLoad=True)   # no LibreOffice on the cluster; Excel calculates on open
    wb.save(out_path)

## 6. Build the Treasury update document

In [0]:
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

NAVY = RGBColor(0x1F, 0x38, 0x64); RED = RGBColor(0xC0, 0x00, 0x00); GREY = RGBColor(0x59, 0x59, 0x59)
PPR_AFTER_BORDER = ("w:shd", "w:tabs", "w:suppressAutoHyphens", "w:kinsoku", "w:wordWrap", "w:overflowPunct",
                    "w:topLinePunct", "w:autoSpaceDE", "w:autoSpaceDN", "w:bidi", "w:adjustRightInd", "w:snapToGrid",
                    "w:spacing", "w:ind", "w:contextualSpacing", "w:mirrorIndents", "w:suppressOverlap", "w:jc",
                    "w:textDirection", "w:textAlignment", "w:textboxTightWrap", "w:outlineLvl", "w:divId",
                    "w:cnfStyle", "w:rPr", "w:sectPr", "w:pPrChange")

def _run(p, text, bold=False, italic=False, size=10, color=None):
    r = p.add_run(text); r.font.name = "Arial"; r._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    r.font.size = Pt(size); r.font.bold = bold; r.font.italic = italic
    if color is not None: r.font.color.rgb = color
    return r

def _para(doc, runs, after=6, before=0, style=None, keep_next=False):
    p = doc.add_paragraph(style=style) if style else doc.add_paragraph()
    p.paragraph_format.space_after = Pt(after); p.paragraph_format.space_before = Pt(before)
    p.paragraph_format.line_spacing = 1.15; p.paragraph_format.keep_with_next = keep_next
    for t in runs:
        if isinstance(t, str): _run(p, t)
        else: _run(p, t[0], **t[1])
    return p

def _heading(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(13); p.paragraph_format.space_after = Pt(5); p.paragraph_format.keep_with_next = True
    _run(p, text, bold=True, size=11, color=NAVY)
    pPr = p._p.get_or_add_pPr(); bdr = OxmlElement("w:pBdr"); b = OxmlElement("w:bottom")
    for k, v in (("w:val", "single"), ("w:sz", "6"), ("w:space", "2"), ("w:color", "2F5597")): b.set(qn(k), v)
    bdr.append(b); pPr.insert_element_before(bdr, *PPR_AFTER_BORDER)
    return p

def _bullet(doc, lead, rest):
    return _para(doc, [(lead, dict(bold=True)), rest], after=4, style="List Bullet")

def _numbered(doc, text):
    return _para(doc, [text], after=4, style="List Number")

def _shade(cell, fill):
    tcPr = cell._tc.get_or_add_tcPr(); shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), fill); tcPr.append(shd)

def _cell(cell, text, bold=False, color=None, size=8.5, align="left", fill=None):
    p = cell.paragraphs[0]; p.paragraph_format.space_after = Pt(0); p.paragraph_format.space_before = Pt(0)
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT if align == "right" else WD_ALIGN_PARAGRAPH.LEFT
    _run(p, text, bold=bold, size=size, color=color)
    if fill: _shade(cell, fill)

def _table(doc, nrows, widths):
    t = doc.add_table(rows=nrows, cols=len(widths)); t.alignment = WD_TABLE_ALIGNMENT.CENTER; t.autofit = False
    tblPr = t._tbl.tblPr; borders = OxmlElement("w:tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        e = OxmlElement(f"w:{edge}")
        for k, v in (("w:val", "single"), ("w:sz", "4"), ("w:space", "0"), ("w:color", "BFBFBF")): e.set(qn(k), v)
        borders.append(e)
    tblPr.insert_element_before(borders, "w:shd", "w:tblLayout", "w:tblCellMar", "w:tblLook", "w:tblCaption", "w:tblDescription")
    for row in t.rows:
        for i, w in enumerate(widths): row.cells[i].width = Inches(w)
        trPr = row._tr.get_or_add_trPr(); cs = OxmlElement("w:cantSplit"); trPr.append(cs)
    return t

def _repeat_header(row):
    trPr = row._tr.get_or_add_trPr(); h = OxmlElement("w:tblHeader"); h.set(qn("w:val"), "true"); trPr.append(h)

def _keep_next_row(row):
    for c in row.cells:
        for p in c.paragraphs: p.paragraph_format.keep_with_next = True

def k0(x): return f"{x:,.0f}" if round(x) >= 0 else f"({abs(x):,.0f})"
def kp(p): return "-" if p is None else (f"{p*100:.1f}%" if p >= 0 else f"({abs(p)*100:.1f}%)")
def lower_first(s): return s[0].lower() + s[1:] if s else s

def line_rows(item): return [r for r in ROWS if r["item"] == item]

def auto_follow_ups():
    """Ask about the two largest weighted drivers. Add quarter-specific items in MANUAL_FOLLOW_UPS."""
    items = []
    for r in sorted(ROWS, key=lambda r: -abs(r["wd"]))[:2]:
        share = r["wd"] / TOTAL_WD if TOTAL_WD else 0
        items.append(f"The source of the {money(r['d23'])} {'increase' if r['d23'] >= 0 else 'decrease'} in Line {r['item']} "
                     f"({r['mat']}), which accounts for roughly {abs(share):.0%} of the quarterly weighted change.")
    return items

def build_docx(out_path):
    doc = Document()
    sec = doc.sections[0]
    sec.left_margin = sec.right_margin = Inches(0.75); sec.top_margin = sec.bottom_margin = Inches(0.75)
    st = doc.styles["Normal"]; st.font.name = "Arial"; st.font.size = Pt(10)
    st.element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    for sname in ("List Bullet", "List Number"):
        doc.styles[sname].font.name = "Arial"; doc.styles[sname].font.size = Pt(10)

    # ----- opening
    _para(doc, [GREETING])
    _para(doc, [f"Below is the quarter-over-quarter Schedule G performance for {Q3}. Supporting detail by line and maturity is "
                f"at the end. Amounts are in $ thousands in the tables and $MM or $B in the commentary."])
    _heading(doc, "Overall Summary")
    up = TOTAL_WD >= 0; word = "rose" if up else "fell"; dirn = "up" if up else "down"
    p1q = TOTAL["q3"] / TOTAL["q1"] - 1
    _para(doc, [f"Total short-term wholesale funding (weighted, item 6) {word} to ",
                (money(TOTAL["q3"]), dict(bold=True)), f" in {Q3}, {dirn} ",
                (f"{money(TOTAL_WD)} ({fpct(TOTAL['q3']/TOTAL['q2']-1)})", dict(bold=True)),
                f" from {money(TOTAL['q2'])} in {Q2} and {'up' if p1q >= 0 else 'down'} {abs(p1q):.0%} from {money(TOTAL['q1'])} in {Q1}."])
    b0 = BUCKET_AGG[MATURITIES[0]]; b0d = b0["q3"] - b0["q2"]; share0 = b0["q3"] / TOTAL["q3"]
    if TOTAL_WD and b0d / TOTAL_WD >= 0.5:
        _para(doc, [f"The {'increase' if up else 'decrease'} was concentrated in the 30-days-or-less maturity bucket, which is about "
                    f"{share0:.0%} of the total and accounted for roughly {money(b0d)} of the quarterly {'growth' if up else 'decline'}."])
    else:
        _para(doc, [f"The 30-days-or-less maturity bucket is about {share0:.0%} of the total and changed by {smoney(b0d)} in the quarter."])
    tier_d = {t: TIER_AGG[t]["q3"] - TIER_AGG[t]["q2"] for t in TIERS}
    lead_tier = max(TIERS, key=lambda t: abs(tier_d[t]))
    top3 = sorted(ROWS, key=lambda r: -abs(r["wd"]))[:3]
    names = [f"Line {r['item']}" for r in top3]
    if len({r["mat"] for r in top3}) == 1:
        drv_txt = ", ".join(names[:-1]) + " and " + names[-1] + f" (all {top3[0]['mat']})"
    else:
        drv_txt = ", ".join(f"{n} ({r['mat']})" for n, r in zip(names[:-1], top3[:-1])) + f" and {names[-1]} ({top3[-1]['mat']})"
    if all(v > 0 for v in tier_d.values()): broad = "Growth was broad across tiers"
    elif all(v < 0 for v in tier_d.values()): broad = "Declines were broad across tiers"
    else: broad = "Tier movements were mixed"
    _para(doc, [f"{broad}, with {lead_tier.title()} the largest contributor. The largest weighted drivers were {drv_txt}."], after=8)

    # tier table
    tw = [2.0, 1.25, 1.25, 1.25, 1.25]
    t = _table(doc, 2 + len(TIERS), tw)
    for i, h in enumerate(["Tier (weighted)", Q2, Q3, "QoQ change", "% of total change"]):
        _cell(t.rows[0].cells[i], h, bold=True, color=RGBColor(255, 255, 255), size=9, align="right" if i else "left", fill="1F3864")
    _repeat_header(t.rows[0])
    for n, tier in enumerate(TIERS):
        a = TIER_AGG[tier]; d = a["q3"] - a["q2"]
        vals = [tier.capitalize(), k0(a["q2"]), k0(a["q3"]), k0(d), kp(d / TOTAL_WD if TOTAL_WD else None)]
        for i, v in enumerate(vals): _cell(t.rows[1 + n].cells[i], v, size=9, align="right" if i else "left")
    vals = ["Total", k0(TOTAL["q2"]), k0(TOTAL["q3"]), k0(TOTAL_WD), "100.0%"]
    for i, v in enumerate(vals): _cell(t.rows[-1].cells[i], v, bold=True, size=9, align="right" if i else "left", fill="D9E2F3")

    # ----- tier sections
    for tier in TIERS:
        _heading(doc, f"{tier.title()} ({smoney(tier_d[tier])} weighted)")
        stable = []
        for _, item, desc, _c in [l for l in LINES if l[0] == tier]:
            lr = line_rows(item); r0 = lr[0]
            pc = pct_chg(r0["q3"], r0["q2"])
            rest = f"{smoney(r0['d23'])} ({fpct(pc)}) to {money(r0['q3'])}"
            if r0["w"] > 0:
                rest += (f", contributing about {smoney(r0['wd'])} weighted." if r0["wd"] >= 0
                         else f", reducing the weighted total by about {money(r0['wd'])}.")
            else:
                rest += "."
            mv = movers(item, [0])
            if mv: rest += " Main movers: " + "; ".join(f"{n} {ud(d)}" for n, d in mv) + "."
            if item in MANUAL_COMMENTARY: rest += " " + MANUAL_COMMENTARY[item]
            _bullet(doc, f"Line {item}, {lower_first(desc)} ({MATURITIES[0]}): ", rest)
            longer = [r for r in lr if r["k"] > 0]
            if max(abs(r["d23"]) for r in longer) >= LONG_BUCKET_MIN_K:
                comb = sum(r["d23"] for r in longer); wcomb = sum(r["wd"] for r in longer)
                per = ", ".join(f"{r['mat']} {ud(r['d23'])}" for r in longer if abs(r["d23"]) >= LONG_BUCKET_MIN_K)
                rest2 = f"{per}. Combined, balances are {ud(comb)} unweighted and {ud(wcomb)} weighted."
                mv2 = movers(item, [1, 2, 3], min_k=25_000, limit=3)
                if mv2: rest2 += " Main movers: " + "; ".join(f"{n} {ud(d)}" for n, d in mv2) + "."
                _bullet(doc, f"Line {item}, longer-dated buckets (31-90, 91-180 and 181-365 days): ", rest2)
            else:
                stable.append(item)
        if stable:
            joined = stable[0] if len(stable) == 1 else ", ".join(stable[:-1]) + " and " + stable[-1]
            _bullet(doc, f"{'Line' if len(stable) == 1 else 'Lines'} {joined}, longer-dated buckets: ",
                    "largely stable, with minimal impact on the weighted total.")

    # ----- follow ups
    _heading(doc, "Follow-Up Items")
    _para(doc, ["Before finalizing the commentary, we should confirm:"], after=4, keep_next=True)
    for txt in auto_follow_ups() + MANUAL_FOLLOW_UPS: _numbered(doc, txt)
    if RWA["q3"] <= 0:
        _numbered(doc, f"The {Q3} average risk-weighted assets figure (item 7), which is not yet in the workbook, so the item 8 ratio cannot be calculated.")
    _para(doc, [SIGN_OFF[0]], before=8, after=0)
    for s_ in SIGN_OFF[1:]: _para(doc, [s_], after=0)

    # ----- detail table
    _heading(doc, "Supporting Detail: Schedule G by Line and Maturity ($ thousands)")
    _para(doc, [("Weight is the Y-15 item 5 maturity weight. Total weighted = sum of weight x balance across all lines. "
                 "Negative changes shown in red parentheses.", dict(italic=True, size=8, color=GREY))], after=6, keep_next=True)
    dw = [1.65, 0.6, 0.9, 0.9, 0.9, 1.0, 1.05]
    n_lines = len(LINES)
    t2 = _table(doc, 1 + n_lines + len(ROWS) + 2, dw)
    for i, h in enumerate(["Remaining maturity", "Weight", Q1, Q2, Q3, "QoQ $ chg", "QoQ % chg"]):
        _cell(t2.rows[0].cells[i], h, bold=True, color=RGBColor(255, 255, 255), size=8.5, align="right" if i else "left", fill="1F3864")
    _repeat_header(t2.rows[0])
    ri = 1
    for tier, item, desc, _c in LINES:
        row = t2.rows[ri]; merged = row.cells[0].merge(row.cells[6])
        _cell(merged, f"{item}  {desc}  ({tier})", bold=True, fill="D9E2F3"); _keep_next_row(row); ri += 1
        for r in line_rows(item):
            d = round(r["q3"]) - round(r["q2"]); pc = pct_chg(r["q3"], r["q2"])
            vals = [r["mat"], f"{r['w']*100:.0f}%", k0(r["q1"]), k0(r["q2"]), k0(r["q3"]), k0(d), kp(pc)]
            for i, v in enumerate(vals):
                _cell(t2.rows[ri].cells[i], v, size=8.5, align="right" if i else "left",
                      color=RED if (i >= 5 and v.startswith("(")) else None)
            ri += 1
    for lab, key, fill in (("Total short-term wholesale funding (weighted, item 6)", "w", "D9E2F3"), ("Total, unweighted gross balances", "u", "F2F2F2")):
        q1_, q2_, q3_ = (TOTAL[q] if key == "w" else UNW[q] for q in ("q1", "q2", "q3"))
        row = t2.rows[ri]; m2 = row.cells[0].merge(row.cells[1])
        _cell(m2, lab, bold=True, fill=fill)
        for i, v in zip(range(2, 7), [k0(q1_), k0(q2_), k0(q3_), k0(round(q3_) - round(q2_)), kp(q3_ / q2_ - 1)]):
            _cell(row.cells[i], v, bold=True, align="right", fill=fill)
        ri += 1
    doc.save(out_path)

## 7. Run

In [0]:
def _write_out(builder, filename):
    local = os.path.join(tempfile.mkdtemp(), filename)
    builder(local)                                   # build on local disk (Volumes do not support random writes)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    dest = os.path.join(OUTPUT_DIR, filename)
    shutil.copy(local, dest)
    return dest

xlsx_path = _write_out(build_excel, f"Schedule_G_{Q3}_QoQ_Analysis.xlsx")
docx_path = _write_out(build_docx, f"Schedule_G_{Q3}_Treasury_Update.docx")
print("Analysis workbook:", xlsx_path)
print("Treasury update  :", docx_path)
print("Download from Catalog Explorer > your volume > output folder.")